# AHR — Worked Hypothetical Examples

ALF-141 · Adversarial Learning Framework

This notebook applies the AHR reference model to two hypothetical scenarios:
an AI lab dealing with scam-generation abuse, and a social media platform
dealing with election-related disinformation. All data below is **synthetic**,
constructed to illustrate how the model works — it does not reflect any real
incident, organization, or measurement.

The model code in the next section is identical to the ALF-141 reference
implementation (`ahr_model.ipynb`); this notebook is self-contained so it can
be run independently.

## Model code

(See `ahr_model.ipynb` for the fully annotated, step-by-step version of this
same code.)

In [ ]:
from dataclasses import dataclass
from typing import Optional
import math

# PLACEHOLDER — Harm Lookup Table: Victim Impact x Scale -> Harm Tier
HARM_LOOKUP_TABLE = {
    ("Low", "Low"): "Low",
    ("Low", "Medium"): "Low",
    ("Low", "High"): "Medium",
    ("Medium", "Low"): "Low",
    ("Medium", "Medium"): "Medium",
    ("Medium", "High"): "High",
    ("High", "Low"): "Medium",
    ("High", "Medium"): "High",
    ("High", "High"): "Critical",
}

# PLACEHOLDER — External Pressure tier definitions
PRESSURE_TIER_DEFINITIONS = {
    "Low": "No known reason for elevated risk.",
    "Medium": "Emerging elevation of risk.",
    "High": "Active sustained pressure requiring ongoing support.",
    "Critical": "Subject to high-profile action likely to cause substantial financial/reputational risk.",
}

# PLACEHOLDER — composite mode
TIER_NUMERIC = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
PRESSURE_MULTIPLIER = {"Low": 1.0, "Medium": 1.15, "High": 1.35, "Critical": 1.6}
TIER_ORDER = ["Low", "Medium", "High", "Critical"]


def percentile_boundaries(historical_values, percentiles=(60, 90), log_transform=True):
    values = [math.log1p(v) for v in historical_values] if log_transform else list(historical_values)
    values = sorted(values)

    def _pctile(p):
        k = (len(values) - 1) * (p / 100)
        f, c = math.floor(k), math.ceil(k)
        if f == c:
            return values[int(k)]
        return values[int(f)] + (values[int(c)] - values[int(f)]) * (k - f)

    return {"low_medium": _pctile(percentiles[0]), "medium_high": _pctile(percentiles[1])}


def tier_from_value(value, boundaries, log_transform=True):
    v = math.log1p(value) if log_transform else value
    if v < boundaries["low_medium"]:
        return "Low"
    if v < boundaries["medium_high"]:
        return "Medium"
    return "High"


# PLACEHOLDER — harm sub-type -> tier reference table (used where a
# categorical reference is available, rather than a continuous proxy)
VICTIM_IMPACT_REFERENCE_TABLE = {
    "scam_investment_elderly": "High",
}


def rate_victim_impact(harm_subtype: str) -> str:
    if harm_subtype not in VICTIM_IMPACT_REFERENCE_TABLE:
        raise KeyError(f"No Victim Impact reference entry for harm sub-type '{harm_subtype}'")
    return VICTIM_IMPACT_REFERENCE_TABLE[harm_subtype]


def rate_scale(raw_values, boundaries, single_indicator_name="exposure", log_transform=True):
    return tier_from_value(raw_values[single_indicator_name], boundaries, log_transform=log_transform)


def rate_external_pressure(tier: str) -> str:
    if tier not in PRESSURE_TIER_DEFINITIONS:
        raise KeyError(f"Unknown pressure tier: {tier!r}")
    return tier


@dataclass
class RatingResult:
    harm_tier: str
    pressure_tier: str
    composite_score: Optional[float] = None

    def as_dict(self) -> dict:
        return {
            "harm_tier": self.harm_tier,
            "pressure_tier": self.pressure_tier,
            "composite_score": round(self.composite_score, 2) if self.composite_score is not None else None,
        }


def rate_incident(victim_impact_tier, scale_tier, pressure_tier, include_composite=False):
    key = (victim_impact_tier, scale_tier)
    if key not in HARM_LOOKUP_TABLE:
        raise KeyError(f"No lookup entry for Victim Impact={victim_impact_tier!r}, Scale={scale_tier!r}")
    harm_tier = HARM_LOOKUP_TABLE[key]

    if pressure_tier not in PRESSURE_TIER_DEFINITIONS:
        raise KeyError(f"Unknown pressure tier: {pressure_tier!r}")

    composite = None
    if include_composite:
        composite = TIER_NUMERIC[harm_tier] * PRESSURE_MULTIPLIER[pressure_tier]

    return RatingResult(harm_tier, pressure_tier, composite)


def rank_queue(incidents, use_composite=False):
    order = {t: i for i, t in enumerate(TIER_ORDER)}
    results = []
    for inc in incidents:
        r = rate_incident(inc["victim_impact_tier"], inc["scale_tier"],
                           inc["pressure_tier"], include_composite=use_composite)
        results.append({"id": inc["id"], **r.as_dict()})
    key = (lambda x: x["composite_score"]) if use_composite else (lambda x: order[x["harm_tier"]])
    return sorted(results, key=key, reverse=True)

---

## Example 1 — AI Lab

> A series of accounts are being used to generate messages for scammers.
> The scam type appears to be investment scams targeting the elderly (both
> factors can form part of the Victim Impact rating), and the indicator
> used for Scale in this case is the quantity of violating responses. The
> jurisdiction in question is the UK, which is rated as Medium for
> External Pressure.

**Victim Impact** — this is a categorical case: the harm sub-type
("investment scam targeting elderly") combines two aggravating factors
(scam type + a recognized vulnerable population) and is rated via the
reference table rather than a continuous proxy. We treat it as **High**.

**Scale** — the organization measures scale by counting violating model
responses generated in support of the scam, rather than downstream
recipient/view counts (which this AI lab may not have visibility into).
Synthetic historical data below represents violating-response counts from
past incidents of this type, used to derive defensible percentile
boundaries rather than a guessed cutoff.

**External Pressure** — given directly as **Medium** (UK jurisdiction).

In [ ]:
# Synthetic historical data: violating response counts from past incidents
# of this type at this AI lab (illustrative only)
historical_violating_responses = [4, 8, 12, 18, 25, 35, 50, 70, 110, 180, 320, 900]

ai_lab_scale_boundaries = percentile_boundaries(historical_violating_responses)
print("AI lab Scale boundaries (log space):", ai_lab_scale_boundaries)

# This incident: 240 violating responses generated by the account network
ai_lab_scale_tier = rate_scale({"exposure": 240}, boundaries=ai_lab_scale_boundaries)
ai_lab_victim_tier = rate_victim_impact("scam_investment_elderly")
ai_lab_pressure_tier = rate_external_pressure("Medium")

print(f"\nVictim Impact tier: {ai_lab_victim_tier}")
print(f"Scale tier: {ai_lab_scale_tier}  (240 violating responses)")
print(f"External Pressure tier: {ai_lab_pressure_tier}")

In [ ]:
ai_lab_result = rate_incident(ai_lab_victim_tier, ai_lab_scale_tier, ai_lab_pressure_tier,
                                include_composite=True)
print("AI lab incident result:", ai_lab_result.as_dict())

---

## Example 2 — Social Media

> A network of 3,000 accounts are posting disinformation content to push a
> specific civic narrative in support of an upcoming election. The Victim
> Impact rating used by this organization is perceived engagement rate (for
> a given number of views, how much engagement is seen). For Scale, the
> measure used is the total number of views across all media/content/ads
> across the complete network.

**Victim Impact** — unlike Example 1, this organization has no categorical
reference table for civic-narrative disinformation; instead it uses a
continuous proxy, **engagement rate** (engagement ÷ views), on the
reasoning that higher engagement per view suggests stronger persuasive
effect on those exposed. This demonstrates that the same percentile
boundary-setting machinery used for Scale can be reused for Victim Impact
when a suitable continuous indicator exists and no categorical reference
is available.

**Scale** — total views across all accounts, content, and ads in the
network. A large network (3,000 accounts) does not by itself imply high
Scale — Scale is measured by actual exposure, not asset count (see
ALF-141 §4).

**External Pressure** — not specified in the scenario; set here as
**High**, reflecting the heightened regulatory and media sensitivity
typically attached to election-related disinformation in the run-up to a
vote. This value is illustrative, not derived, and would ordinarily be
supplied by the policy/legal function.

In [ ]:
# Synthetic historical data: engagement rate (engagement / views) for past
# civic-narrative disinformation incidents at this platform (illustrative only)
historical_engagement_rates = [0.003, 0.006, 0.010, 0.015, 0.020, 0.028,
                                0.040, 0.055, 0.075, 0.095, 0.130, 0.180]

social_media_victim_boundaries = percentile_boundaries(historical_engagement_rates)
print("Social media Victim Impact boundaries (log space):", social_media_victim_boundaries)

# This incident: an observed engagement rate of 0.09 (9%)
social_media_victim_tier = tier_from_value(0.09, social_media_victim_boundaries)
print(f"\nVictim Impact tier: {social_media_victim_tier}  (engagement rate 0.09)")

In [ ]:
# Synthetic historical data: total network views for past civic-narrative
# disinformation incidents at this platform (illustrative only)
historical_network_views = [2_000, 15_000, 60_000, 150_000, 400_000, 900_000,
                             2_000_000, 4_500_000, 9_000_000, 20_000_000]

social_media_scale_boundaries = percentile_boundaries(historical_network_views)
print("Social media Scale boundaries (log space):", social_media_scale_boundaries)

# This incident: 2,300,000 total views across the 3,000-account network
social_media_scale_tier = rate_scale({"exposure": 2_300_000}, boundaries=social_media_scale_boundaries)
print(f"\nScale tier: {social_media_scale_tier}  (2,300,000 total views)")

social_media_pressure_tier = rate_external_pressure("High")
print(f"External Pressure tier: {social_media_pressure_tier}")

In [ ]:
social_media_result = rate_incident(social_media_victim_tier, social_media_scale_tier,
                                      social_media_pressure_tier, include_composite=True)
print("Social media incident result:", social_media_result.as_dict())

---

## Comparing the two incidents

Stack-ranking both hypothetical incidents together, the same way an
analyst would triage a mixed queue across harm types.

In [ ]:
queue = [
    {
        "id": "AI-LAB-2201",
        "victim_impact_tier": ai_lab_victim_tier,
        "scale_tier": ai_lab_scale_tier,
        "pressure_tier": ai_lab_pressure_tier,
    },
    {
        "id": "SOCIAL-4417",
        "victim_impact_tier": social_media_victim_tier,
        "scale_tier": social_media_scale_tier,
        "pressure_tier": social_media_pressure_tier,
    },
]

print("Ranked by Harm Tier (default triage order):\n")
for row in rank_queue(queue, use_composite=False):
    print(row)

print("\nRanked by Composite Score (volume-triage mode, decomposable):\n")
for row in rank_queue(queue, use_composite=True):
    print(row)

**Note:** as covered in ALF-141 §6 (Limitations), this side-by-side
ranking is illustrative of the mechanics only. Comparing a scam-generation
incident against a disinformation incident is an inter-harm comparison,
which this model does not claim to resolve on severity grounds alone —
see the Limitations section for how such cross-domain decisions are
typically handled in practice.